In [2]:
from pathlib import Path
from safetensors import safe_open
from transformers.utils.hub import cached_file
from transformers import AutoConfig
import torch
import json
import os
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

/home/jeromeku/verl/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MODEL_ID = "Qwen/Qwen3-0.6B"
model_cache_dir = os.path.dirname(cached_file(MODEL_ID, filename="config.json"))

In [4]:
index_file = "model.safetensors.index.json"
has_index_file = os.path.exists(os.path.join(os.path.join(model_cache_dir, index_file)))


In [5]:
model_file = Path(model_cache_dir) / "model.safetensors"
with safe_open(model_file, 'pt', 'cpu') as f:
    qkvs = {k: f.get_tensor(k) for k in f.keys() if 'layers.0' in k and any(p in k for p in ['q_proj', 'k_proj', 'v_proj'])}

In [6]:
config = AutoConfig.from_pretrained(MODEL_ID)

In [7]:
qkvs.keys()

dict_keys(['model.layers.0.self_attn.k_proj.weight', 'model.layers.0.self_attn.q_proj.weight', 'model.layers.0.self_attn.v_proj.weight'])

In [ ]:
def get_qkv():
    q = qkvs.get("model.layers.0.self_attn.q_proj.weight")
    k = qkvs.get("model.layers.0.self_attn.k_proj.weight")
    v = qkvs.get("model.layers.0.self_attn.v_proj.weight")
    return q, k, v

In [13]:
def make_arange(t: torch.Tensor, dtype=None):
  t = t.clone()
  t_arr = torch.arange(t.numel(), dtype=dtype or t.dtype).reshape_as(t)
  return t_arr
q = make_arange(q, torch.float)
k = make_arange(k, torch.float)
v = make_arange(v, torch.float)

tensor([[0.0000e+00, 1.0000e+00, 2.0000e+00, 3.0000e+00, 4.0000e+00],
        [1.0240e+03, 1.0250e+03, 1.0260e+03, 1.0270e+03, 1.0280e+03],
        [2.0480e+03, 2.0490e+03, 2.0500e+03, 2.0510e+03, 2.0520e+03],
        [3.0720e+03, 3.0730e+03, 3.0740e+03, 3.0750e+03, 3.0760e+03],
        [4.0960e+03, 4.0970e+03, 4.0980e+03, 4.0990e+03, 4.1000e+03]])

In [28]:
num_key_value_heads = config.num_key_value_heads
hidden_dim = config.hidden_size
num_attention_heads = config.num_attention_heads
head_dim = getattr(
    config, "head_dim", hidden_dim // num_attention_heads
)
group_dim = head_dim * num_attention_heads // num_key_value_heads
num_key_value_heads, num_attention_heads, head_dim, group_dim
assert q.shape[0] == num_attention_heads * head_dim
q_proj_size = q.shape[0]
kv_proj_size = k.shape[0]
q_proj_size, kv_proj_size

(8, 16, 128, 256)

(2048, 1024)

In [ ]:
def merge_qkv(q, k, v, config):
    q, k, v = q.clone(), k.clone(), v.clone()
    
    num_key_value_heads = config.num_key_value_heads
    hidden_dim = config.hidden_size
    num_attention_heads = config.num_attention_heads

    head_dim = getattr(
        config, "head_dim", hidden_dim // num_attention_heads
    )

    q_kv_shape = q.view(num_key_value_heads, -1, hidden_dim).shape
    group_dim = head_dim * num_attention_heads // num_key_value_heads # query dim if num_kv_heads q heads
    assert q_kv_shape[1] == group_dim

    q_proj_size = q.shape[0]
    kv_proj_size = k.shape[0]
    assert q_proj_size == head_dim * num_attention_heads
    assert kv_proj_size == head_dim * num_key_value_heads

    real_num_key_value_heads = q.shape[0] // group_dim
    assert real_num_key_value_heads == num_key_value_heads

    q = q.view(
        [
            num_key_value_heads,
            group_dim,
            -1,
        ]
    )
    assert q_kv_shape == q.shape

    k = k.view([num_key_value_heads, head_dim, -1])
    v = v.view([num_key_value_heads, head_dim, -1])
    out_shape = [-1, hidden_dim]

    qkv = torch.cat([q, k, v], dim=1).view(*out_shape).contiguous()
    
    return qkv